# Questão 4 - Dados Públicos
### Cenário

O Sr. Almir identificou que alguns produtos podem ter sido vendidos abaixo do custo, possivelmente por erro operacional.

O problema é que:

- O sistema de vendas (`vendas_2023_2024.csv`) registra valores em **real BRL**
- O catálogo de fornecedores (`custos_importacao.json`) registra custos unitarios em **dollar USD**
- O câmbio varia diariamente

Até hoje, ninguem cruzou o custo em dolar do dia da venda com o valor de venda em reais.

Sua missão é abrir essa "caixa preta" financeira e identificar onde houve prejuizo real.

### Premissas obrigatórias:

- O custo em USD é unitário
- O custo em BRL deve ser calculado usando o cambio da data da venda
- A taxa de cámbio deve ser considerada a média da cotação de venda do dia (Banco Central)
- A receita total do produto considera todas as vendas (inclusive as sem prejuizo)
- Ignore impostos e frete

### Tarefas:

> Parte 1 - Calculo e Modelagem

- Calcule o custo total em BRL por transação
- Identifique transações com prejuízo
- Agregue os dados por id_produto, gerando:
1. Receita total (BRL)
2. Prejuizo total (BRL)
3. Percentual de perda (prejuízo_total / receita_total)

> Parte 2- Análise Visual

* Gere um gráfico que represente o prejuízo total por produto, considerando apenas produtos que tiveram prejuizo. *(inserir o gráfico no relatório/dashboard final)*

> Parte 3 - Análise Objetiva 

Responda objetivamente:

- Qual produto concentra o maior prejulzo absoluto?
- O produto com maior prejuizo absoluto também e o que possui a maior porcentagem de perda? (Sim ou Não)

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

In [ ]:
ancoras = {
    "2023-01-02": 5.2851, "2023-01-31": 5.0784, "2023-02-28": 5.2326,
    "2023-03-31": 5.0799, "2023-04-28": 5.0612, "2023-05-31": 4.9921,
    "2023-06-30": 4.8216, "2023-07-31": 4.7310, "2023-08-31": 4.9782,
    "2023-09-29": 4.9987, "2023-10-31": 5.0286, "2023-11-30": 4.9261,
    "2023-12-29": 4.8477, "2024-01-31": 4.9703, "2024-02-29": 4.9735,
    "2024-03-28": 4.9982, "2024-04-30": 5.1765, "2024-05-31": 5.2086,
    "2024-06-28": 5.3897, "2024-07-31": 5.6437, "2024-08-30": 5.4450,
    "2024-09-30": 5.4425, "2024-10-31": 5.7841, "2024-11-29": 6.0498,
    "2024-12-31": 6.1805,
}

s = pd.Series(ancoras)
s.index = pd.to_datetime(s.index)
all_days = pd.date_range("2023-01-01", "2024-12-31", freq="D")
cambio = s.reindex(all_days).interpolate(method="linear")
cambio_dict = cambio.to_dict()

In [ ]:
# ajuste o caminho para onde seus arquivos estão
PASTA = "../datasets/"

vendas = pd.read_csv(PASTA + "vendas_2023_2024.csv")

with open(PASTA + "custos_importacao.json", encoding="utf-8") as f:
    custos_raw = json.load(f)

In [ ]:
def parse_date(d):
    for fmt in ("%Y-%m-%d", "%d-%m-%Y"):
        try:
            return pd.to_datetime(d, format=fmt)
        except:
            pass
    return pd.NaT

vendas["sale_date"] = vendas["sale_date"].apply(parse_date)

In [ ]:
linhas = []
for prod in custos_raw:
    for reg in prod["historic_data"]:
        linhas.append({
            "product_id":   prod["product_id"],
            "product_name": prod["product_name"],
            "start_date":   pd.to_datetime(reg["start_date"], format="%d/%m/%Y"),
            "usd_price":    reg["usd_price"],
        })

custos = pd.DataFrame(linhas).sort_values(["product_id", "start_date"])

In [ ]:
vendas["usd_brl"]         = vendas["sale_date"].map(cambio_dict)
vendas["custo_total_brl"] = vendas["usd_price"] * vendas["usd_brl"] * vendas["qtd"]
vendas["receita"]         = vendas["total"]
vendas["resultado"]       = vendas["receita"] - vendas["custo_total_brl"]
vendas["prejuizo"]        = vendas["resultado"].apply(lambda x: x if x < 0 else 0)

vendas.head()

In [ ]:
prod_names = {p["product_id"]: p["product_name"] for p in custos_raw}

agg = vendas.groupby("id_product").agg(
    receita_total  = ("receita",   "sum"),
    prejuizo_total = ("prejuizo",  "sum"),
    n_vendas       = ("id",        "count"),
).reset_index()

agg["pct_perda"]    = (agg["prejuizo_total"].abs() / agg["receita_total"] * 100).round(2)
agg["product_name"] = agg["id_product"].map(prod_names)
agg = agg.sort_values("prejuizo_total")

prej = agg[agg["prejuizo_total"] < 0].copy()
prej.head(10)

In [ ]:
top20 = prej.head(20).copy()
top20["prejuizo_abs"] = top20["prejuizo_total"].abs()
top20["label"] = top20["product_name"].str.replace(
    r"Motor (de Popa |Elétrico |Diesel )?", "", regex=True
).str[:30]

fig, ax = plt.subplots(figsize=(12, 8))
colors = ["#ff5252" if i == 0 else "#ff7070" if i < 3 else "#c94040"
          for i in range(len(top20))]

ax.barh(range(len(top20)), top20["prejuizo_abs"], color=colors, height=0.7)

for i, row in enumerate(top20.itertuples()):
    ax.text(row.prejuizo_abs + 100_000, i,
            f"R$ {row.prejuizo_abs/1e6:.1f}M  ({row.pct_perda:.1f}%)",
            va="center", fontsize=8)

ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20["label"], fontsize=9)
ax.invert_yaxis()
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R${x/1e6:.0f}M"))
ax.set_title("Top 20 Produtos com Maior Prejuízo — LH Nautical 2023–2024")
ax.set_xlabel("Prejuízo Total (BRL)")
plt.tight_layout()
plt.show()

In [ ]:
maior_abs = prej.loc[prej["prejuizo_total"].idxmin()]
maior_pct = prej.loc[prej["pct_perda"].idxmax()]

print("MAIOR PREJUÍZO ABSOLUTO")
print(f"  Produto   : {maior_abs['product_name']}")
print(f"  Prejuízo  : R$ {maior_abs['prejuizo_total']:,.2f}")
print(f"  % de perda: {maior_abs['pct_perda']:.2f}%")

print("\nMAIOR % DE PERDA")
print(f"  Produto   : {maior_pct['product_name']}")
print(f"  % de perda: {maior_pct['pct_perda']:.2f}%")

mesmo = maior_abs["id_product"] == maior_pct["id_product"]
print(f"\nSão o mesmo produto? {'Sim ✓' if mesmo else 'Não ✗'}")

gemini

In [1]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
CAMBIO_MEDIO = 5.15

In [5]:
# carregar os dados
vendas = pd.read_csv('../datasets/vendas_normalizadas.csv')
produtos = pd.read_csv('../datasets/produtos_limpos.csv')
custos_importacao = pd.read_csv('../datasets/custos_importacao.csv')

In [19]:
# Flatten custos_importacao
custo_lista = []

for p in custos_importacao:
    for h in p['historic_data']:
        # 1. Limpamos qualquer espaço em branco invisível
        data_texto = str(h['start_date']).strip()
        
        custo_lista.append({
            'id_product': p['product_id'],
            'usd_price': h['usd_price'],
            # 2. Usamos o FORMATO EXPLICITO em vez de dayfirst. 
            # Isso é 'blindado' contra erros de interpretação.
            'data_custo': pd.to_datetime(data_texto, format='%d/%m/%Y', errors='coerce')
        })

df_custos = pd.DataFrame(custo_lista).sort_values('data_custo')

# 3. Verificação (Opcional, mas recomendada pelo Gabriel)
print(f"Datas com erro (NaT): {df_custos['data_custo'].isna().sum()}")
display(df_custos.head())

TypeError: string indices must be integers, not 'str'